# Clase 7 · Evaluar y regularizar modelos

**Módulo 1: Introducción y fundamentos estadísticos** · Diplomado en Ciencia de Datos Aplicada · UTFSM

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daniopitz/diplomado-cdd/blob/main/07_evaluar_regularizar_colab.ipynb)

Versión para **Google Colab**. Antes de trabajar, guarde su propia copia con `Archivo > Guardar una copia en Drive`.

**Qué hacemos hoy.** Dos mitades. En la primera volvemos a los 10.000 clientes de tarjeta de crédito de la clase 6 y medimos por separado los dos errores del clasificador (sensibilidad, especificidad y precisión), elegimos el umbral según el costo de cada error y comparamos modelos con la curva ROC. En la segunda volvemos a la regresión lineal con los salarios de 263 jugadores de béisbol y 19 variables: vemos cómo mínimos cuadrados se ajusta al azar de su muestra, y cómo Ridge y Lasso lo corrigen. Los dos ejemplos son del libro de James, Witten, Hastie y Tibshirani.

**El hilo de la clase:** un modelo se juzga por sus errores donde importan, con el costo correcto y en datos que no vio.

## 1. Preparación

Las bibliotecas de siempre, más **scikit-learn** (se importa como `sklearn`), la biblioteca de aprendizaje automático de Python. Hoy la usamos para la curva ROC, para separar los datos, para estandarizar y para Ridge y Lasso.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

# scikit-learn: la biblioteca de aprendizaje automático de Python
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV

sns.set_theme(style="whitegrid")
%config InlineBackend.figure_format = "retina"
pd.options.display.float_format = "{:.3f}".format

In [ ]:
# Los datos se leen directo desde el repositorio del curso en GitHub
BASE = "https://raw.githubusercontent.com/daniopitz/diplomado-cdd/main/datos/"

## 2. Recordatorio: la matriz de confusión de la clase 6

**Objetivo:** volver a tener el modelo de la deuda y su matriz de confusión, que es de donde salen todas las medidas de hoy.

Los mismos pasos de la clase 6: la columna `no_paga` vale 1 si el cliente dejó de pagar, la deuda en cientos de dólares, y la regresión logística.

In [ ]:
clientes = pd.read_csv(BASE + "clase06/default_islr.csv")
clientes["no_paga"] = (clientes["default"] == "Yes").astype(int)
clientes["deuda_100"] = clientes["balance"] / 100
clientes.head()

In [ ]:
logistica = smf.logit("no_paga ~ deuda_100", data=clientes).fit(disp=0)
clientes["prob"] = logistica.predict(clientes)                  # probabilidad de no pagar
clientes["prediccion"] = (clientes["prob"] >= 0.5).astype(int)  # umbral 0,5
clientes[["balance", "no_paga", "prob", "prediccion"]].head()

In [ ]:
# La matriz de confusión: lo real en las filas, lo predicho en las columnas
matriz = pd.crosstab(clientes["no_paga"], clientes["prediccion"], rownames=["real"], colnames=["predicho"])
matriz

Para calcular las medidas conviene tener las cuatro celdas como números. `matriz.loc[fila, columna]` toma una celda: la fila es lo real y la columna lo predicho, así que `matriz.loc[1, 1]` son los clientes que no pagaron y el modelo declaró morosos.

In [ ]:
VP = matriz.loc[1, 1]   # verdadero positivo: no pagó y el modelo dijo "no paga"
FN = matriz.loc[1, 0]   # falso negativo: no pagó y el modelo dijo "paga"
FP = matriz.loc[0, 1]   # falso positivo: pagó y el modelo dijo "no paga"
VN = matriz.loc[0, 0]   # verdadero negativo: pagó y el modelo dijo "paga"
print(f"VP = {VP}, FN = {FN}, FP = {FP}, VN = {VN}")

In [ ]:
exactitud = (VP + VN) / len(clientes)
base = 1 - clientes["no_paga"].mean()   # decir siempre "paga"
print(f"exactitud del modelo: {exactitud:.3f}")
print(f"decir siempre \"paga\": {base:.3f}")

Casi lo mismo: la exactitud mezcla los dos errores y, como el sí es raro, queda dominada por los que pagan. Hoy medimos cada error por separado.

## 3. Sensibilidad, especificidad y precisión

**Objetivo:** medir cada error por separado con tres fracciones de la matriz.

**Sensibilidad** (en inglés, *recall*): de los clientes que no pagan, ¿qué fracción detecta el modelo? Mira la fila de los que no pagan:

$$\text{sensibilidad} = \frac{VP}{VP + FN}$$

In [ ]:
sensibilidad = VP / (VP + FN)
print(f"sensibilidad = {VP} / ({VP} + {FN}) = {sensibilidad:.2f}")

El modelo detecta 3 de cada 10 clientes que no pagan.

**Especificidad**: de los clientes que pagan, ¿qué fracción deja en paz? Mira la fila de los que pagan:

$$\text{especificidad} = \frac{VN}{VN + FP}$$

In [ ]:
especificidad = VN / (VN + FP)
print(f"especificidad = {VN} / ({VN} + {FP}) = {especificidad:.3f}")

**Precisión** (en inglés, *precision*): de los clientes que el modelo declara morosos, ¿qué fracción lo es? Mira la columna del predicho "no paga":

$$\text{precisión} = \frac{VP}{VP + FP}$$

In [ ]:
precision = VP / (VP + FP)
print(f"precisión = {VP} / ({VP} + {FP}) = {precision:.2f}")

De cada 10 clientes que el modelo declara morosos, 7 lo son. Resumen: el modelo casi no molesta a los que pagan (especificidad 0,996), cuando marca a alguien suele acertar (precisión 0,70), pero se le escapan 7 de cada 10 morosos (sensibilidad 0,30).

## 4. Mover el umbral

**Objetivo:** ver qué pasa con los errores si se dice "no paga" desde otra probabilidad.

El modelo no cambia: solo cambia el corte. Con umbral 0,2:

In [ ]:
prediccion_02 = (clientes["prob"] >= 0.2).astype(int)
pd.crosstab(clientes["no_paga"], prediccion_02, rownames=["real"], colnames=["predicho"])

Se detectan 199 morosos en vez de 100, a cambio de 263 falsos positivos en vez de 42. Para repetir el cálculo con varios umbrales sin copiar código, una función que recibe un umbral y devuelve las cuatro celdas y las medidas. Hace exactamente los pasos anteriores, contando cada celda con una condición:

In [ ]:
def medidas(umbral):
    """Las cuatro celdas de la matriz y las medidas, para un umbral."""
    predicho = (clientes["prob"] >= umbral).astype(int)
    real = clientes["no_paga"]
    VP = ((real == 1) & (predicho == 1)).sum()
    FN = ((real == 1) & (predicho == 0)).sum()
    FP = ((real == 0) & (predicho == 1)).sum()
    VN = ((real == 0) & (predicho == 0)).sum()
    return {"umbral": umbral, "VP": VP, "FN": FN, "FP": FP, "VN": VN,
            "sensibilidad": VP / (VP + FN), "especificidad": VN / (VN + FP)}

medidas(0.5)

In [ ]:
tabla_umbrales = pd.DataFrame([medidas(u) for u in [0.5, 0.3, 0.2, 0.1]])
tabla_umbrales

Con todos los umbrales entre 0,01 y 0,98 se ve el balance completo: bajar el umbral sube la sensibilidad y baja la especificidad.

In [ ]:
umbrales = np.round(np.arange(0.01, 0.99, 0.01), 2)
curva = pd.DataFrame([medidas(u) for u in umbrales])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(curva["umbral"], curva["sensibilidad"], lw=2.5, label="sensibilidad")
ax.plot(curva["umbral"], curva["especificidad"], lw=2.5, label="especificidad")
ax.set(xlabel="Umbral", ylabel="Fracción", title="Sensibilidad y especificidad según el umbral")
ax.legend()
plt.show()

## 5. El costo de cada error

**Objetivo:** elegir el umbral según lo que cuesta cada error.

**Supuesto** (en un proyecto real lo define el problema): darle la tarjeta a un cliente que no paga, un falso negativo, cuesta 10; negársela a uno que sí paga, un falso positivo, cuesta 1. El costo total de un umbral es:

$$\text{costo total} = 10 \cdot FN + 1 \cdot FP$$

In [ ]:
costo_fn = 10
costo_fp = 1
tabla_umbrales["costo"] = costo_fn * tabla_umbrales["FN"] + costo_fp * tabla_umbrales["FP"]
tabla_umbrales[["umbral", "FN", "FP", "costo"]]

El umbral 0,5 es el más caro de los cuatro. Con todos los umbrales, el de menor costo:

In [ ]:
curva["costo"] = costo_fn * curva["FN"] + costo_fp * curva["FP"]
mejor = curva.loc[curva["costo"].idxmin()]   # la fila con el menor costo
print(f"umbral de menor costo: {mejor['umbral']:.2f}, con costo {mejor['costo']:.0f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(curva["umbral"], curva["costo"], lw=2.5, color="orange")
ax.axvline(mejor["umbral"], color="red", ls="--")
ax.set(xlabel="Umbral", ylabel="Costo total", title="Costo total según el umbral (FN cuesta 10, FP cuesta 1)")
plt.show()

**De dónde sale ese umbral.** Para un cliente con probabilidad $P$ de no pagar, aprobarlo cuesta en promedio $P \cdot 10$ (si no paga, cuesta 10) y rechazarlo cuesta $(1 - P) \cdot 1$ (si iba a pagar, cuesta 1). Conviene rechazar cuando lo primero supera a lo segundo, y despejando $P$ queda la regla:

$$\text{umbral} = \frac{\text{costo FP}}{\text{costo FP} + \text{costo FN}}$$

In [ ]:
print(f"costos 10 y 1: umbral = 1 / (1 + 10) = {costo_fp / (costo_fp + costo_fn):.3f}")
print(f"costos iguales: umbral = 1 / (1 + 1) = {1 / (1 + 1):.3f}")

La regla da 0,09, cerca del 0,08 que encontró la búsqueda. Y con costos iguales da 0,5: el umbral por defecto supone que los dos errores cuestan lo mismo.

## 6. La curva ROC y el AUC

**Objetivo:** comparar clasificadores sin tener que elegir un umbral.

La **curva ROC** dibuja, para cada umbral, la sensibilidad contra 1 − especificidad (la fracción de los que pagan que el modelo declara morosos). `roc_curve` recibe lo real y la probabilidad, y devuelve las dos coordenadas para todos los umbrales:

In [ ]:
x_roc, y_roc, umbrales_roc = roc_curve(clientes["no_paga"], clientes["prob"])
# x_roc: 1 − especificidad; y_roc: sensibilidad; un valor por umbral

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(x_roc, y_roc, lw=2.5, color="orange", label="modelo con la deuda")
ax.plot([0, 1], [0, 1], "--", color="gray", label="adivinar al azar")
ax.set(xlabel="1 − especificidad", ylabel="Sensibilidad", title="Curva ROC")
ax.legend()
plt.show()

El **AUC** (área bajo la curva) resume la curva en un número entre 0,5 (adivinar) y 1 (separar perfecto). Se lee así: si se toma al azar un cliente que no paga y uno que paga, el AUC es la fracción de esos pares en que el modelo le da más probabilidad al que no paga.

In [ ]:
auc_deuda = roc_auc_score(clientes["no_paga"], clientes["prob"])
print(f"AUC del modelo con la deuda: {auc_deuda:.3f}")

Para ver que el AUC compara modelos, uno mucho peor: solo con la variable estudiante.

In [ ]:
clientes["estudiante"] = (clientes["student"] == "Yes").astype(int)
solo_estudiante = smf.logit("no_paga ~ estudiante", data=clientes).fit(disp=0)
clientes["prob_estudiante"] = solo_estudiante.predict(clientes)

auc_estudiante = roc_auc_score(clientes["no_paga"], clientes["prob_estudiante"])
print(f"AUC del modelo solo con estudiante: {auc_estudiante:.3f}")

In [ ]:
x_est, y_est, _ = roc_curve(clientes["no_paga"], clientes["prob_estudiante"])

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(x_roc, y_roc, lw=2.5, color="orange", label=f"con la deuda (AUC {auc_deuda:.2f})")
ax.plot(x_est, y_est, lw=2.5, color="purple", label=f"solo con estudiante (AUC {auc_estudiante:.2f})")
ax.plot([0, 1], [0, 1], "--", color="gray", label="adivinar al azar")
ax.set(xlabel="1 − especificidad", ylabel="Sensibilidad", title="Dos modelos, dos curvas ROC")
ax.legend()
plt.show()

## 7. Otro problema: el salario de jugadores de béisbol

**Objetivo:** tener un problema de regresión con muchas variables.

Los datos `Hitters` son del laboratorio de Ridge y Lasso del libro (sección 6.6): jugadores de las grandes ligas de béisbol de Estados Unidos, con sus estadísticas de la temporada 1986 y de toda su carrera, y su salario de 1987 en miles de dólares. Las columnas vienen en inglés; las renombramos:

| original | nombre nuevo | qué es |
|---|---|---|
| AtBat, Hits, HmRun | turnos, hits, jonrones | turnos al bate, hits y jonrones en 1986 |
| Runs, RBI, Walks | anotadas, impulsadas, bases_bolas | carreras anotadas e impulsadas y bases por bolas en 1986 |
| Years | temporadas | años en las grandes ligas |
| CAtBat, CHits, CHmRun, CRuns, CRBI, CWalks | los mismos con `_carrera` | los mismos totales, acumulados en toda la carrera |
| PutOuts, Assists, Errors | outs_defensa, asistencias, errores | jugadas en defensa en 1986 |
| League, Division, NewLeague | liga_nacional, division_oeste, liga_nacional_1987 | dummies: liga y división en 1986, y liga en 1987 |
| Salary | salario | la respuesta, en miles de dólares |

In [ ]:
jugadores = pd.read_csv(BASE + "clase07/hitters_islr.csv")
print("filas y columnas:", jugadores.shape)
print("jugadores sin salario:", jugadores["Salary"].isna().sum())
jugadores.head()

Se quitan los jugadores sin salario, porque el salario es lo que queremos explicar, y se renombran las columnas.

In [ ]:
jugadores = jugadores.dropna(subset=["Salary"])

nombres = {
    "AtBat": "turnos", "Hits": "hits", "HmRun": "jonrones", "Runs": "anotadas", "RBI": "impulsadas",
    "Walks": "bases_bolas", "Years": "temporadas", "CAtBat": "turnos_carrera", "CHits": "hits_carrera",
    "CHmRun": "jonrones_carrera", "CRuns": "anotadas_carrera", "CRBI": "impulsadas_carrera",
    "CWalks": "bases_bolas_carrera", "PutOuts": "outs_defensa", "Assists": "asistencias", "Errors": "errores",
}
X = jugadores.rename(columns=nombres)[list(nombres.values())].astype(float)
X["liga_nacional"] = (jugadores["League"] == "N").astype(float)
X["division_oeste"] = (jugadores["Division"] == "W").astype(float)
X["liga_nacional_1987"] = (jugadores["NewLeague"] == "N").astype(float)
y = jugadores["Salary"]

print("jugadores:", len(X), "| variables:", X.shape[1])
X.head()

Muchas de estas variables dicen casi lo mismo: un jugador con más temporadas tiene más de todo en su carrera.

In [ ]:
X[["temporadas", "turnos_carrera", "hits_carrera"]].corr()

## 8. Entrenamiento, prueba y el RMSE

**Objetivo:** medir el error de mínimos cuadrados en datos que no vio.

Como en la clase 6, se separan los datos al azar antes de ajustar. `train_test_split` hace el reparto: `test_size=0.5` deja la mitad para prueba, como en el laboratorio del libro, y `random_state=1` fija la semilla para que el reparto sea siempre el mismo.

In [ ]:
X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.5, random_state=1)
print("entrenamiento:", len(X_ent), "jugadores | prueba:", len(X_pru), "jugadores")

**Estandarizar.** Antes de comparar coeficientes, y sobre todo antes de penalizarlos, todas las variables deben estar en la misma escala: a cada una se le resta su media y se divide por su desviación estándar,

$$z = \frac{x - \text{media}}{\text{desviación estándar}}$$

`StandardScaler` hace esa cuenta. Con `fit` calcula la media y la desviación **solo con los datos de entrenamiento**, y con `transform` la aplica, a entrenamiento y a prueba por igual.

In [ ]:
escala = StandardScaler().fit(X_ent)
Z_ent = pd.DataFrame(escala.transform(X_ent), columns=X.columns, index=X_ent.index)
Z_pru = pd.DataFrame(escala.transform(X_pru), columns=X.columns, index=X_pru.index)

# Después de estandarizar, cada variable de entrenamiento tiene media 0 y desviación 1
Z_ent.describe().loc[["mean", "std"]].round(2)

**Mínimos cuadrados en scikit-learn.** `LinearRegression` es la misma regresión de la clase 4. La forma de usar cualquier modelo de scikit-learn es siempre la misma: `fit(X, y)` lo ajusta, `predict(X)` predice y `coef_` guarda los coeficientes.

In [ ]:
mco = LinearRegression().fit(Z_ent, y_ent)
coef_mco = pd.Series(mco.coef_, index=X.columns)
coef_mco.sort_values().round(0).astype(int)

Turnos al bate y hits de la carrera, dos variables que dicen casi lo mismo, reciben coeficientes enormes y de signo opuesto: se cancelan entre sí. Es la señal de un modelo que se ajusta al azar de sus 131 jugadores.

**El RMSE** (raíz del error cuadrático medio) mide el error de una regresión: se promedian los residuos al cuadrado y se saca la raíz, para volver a las unidades de y (miles de dólares):

In [ ]:
def rmse(real, predicho):
    return np.sqrt(np.mean((real - predicho) ** 2))

print(f"mínimos cuadrados, en entrenamiento: {rmse(y_ent, mco.predict(Z_ent)):.0f}")
print(f"mínimos cuadrados, en prueba:        {rmse(y_pru, mco.predict(Z_pru)):.0f}")
print(f"sin modelo (la media), en prueba:    {rmse(y_pru, y_ent.mean()):.0f}")

Se equivoca mucho más con los jugadores que no vio. El error que vale es el de prueba.

## 9. Ridge

**Objetivo:** castigar los coeficientes grandes y ver qué pasa con el error en prueba.

Ridge busca los coeficientes que minimizan

$$\sum_i (y_i - \hat{y}_i)^2 + \lambda \sum_j b_j^2$$

donde el primer término es el ajuste de mínimos cuadrados, $b_j$ es el coeficiente de la variable $j$ y $\lambda$ (lambda) decide cuánto pesa el castigo. Con $\lambda = 0$ es mínimos cuadrados; con $\lambda$ muy grande, todos los coeficientes van a 0. En scikit-learn, $\lambda$ se llama `alpha`. Con $\lambda = 100$:

In [ ]:
ridge = Ridge(alpha=100).fit(Z_ent, y_ent)
comparacion = pd.DataFrame({
    "mínimos cuadrados": mco.coef_,
    "Ridge (λ = 100)": ridge.coef_,
}, index=X.columns)
comparacion.round(0).astype(int)

In [ ]:
print(f"Ridge (λ = 100), en prueba: {rmse(y_pru, ridge.predict(Z_pru)):.0f}")

Los coeficientes opuestos se achican y se juntan, y el error en prueba baja. El **camino de Ridge**: los coeficientes para muchos valores de $\lambda$, de casi 0 a muy grande.

In [ ]:
lambdas = np.logspace(-4, 5, 60)   # 60 valores de 0,0001 a 100.000, en escala logarítmica
camino_ridge = pd.DataFrame([Ridge(alpha=l).fit(Z_ent, y_ent).coef_ for l in lambdas],
                            index=lambdas, columns=X.columns)

fig, ax = plt.subplots(figsize=(9, 4.5))
for variable in X.columns:
    grueso = variable in ["turnos_carrera", "hits_carrera"]
    ax.plot(camino_ridge.index, camino_ridge[variable], lw=3 if grueso else 1,
            label=variable if grueso else None)
ax.set_xscale("log")
ax.set(xlabel="λ (escala logarítmica)", ylabel="Coeficiente estandarizado", title="El camino de Ridge")
ax.legend()
plt.show()

Y el error en entrenamiento y en prueba según $\lambda$:

In [ ]:
error_ent = [rmse(y_ent, Ridge(alpha=l).fit(Z_ent, y_ent).predict(Z_ent)) for l in lambdas]
error_pru = [rmse(y_pru, Ridge(alpha=l).fit(Z_ent, y_ent).predict(Z_pru)) for l in lambdas]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(lambdas, error_ent, lw=2.5, label="entrenamiento")
ax.plot(lambdas, error_pru, lw=2.5, label="prueba")
ax.set_xscale("log")
ax.set(xlabel="λ (escala logarítmica)", ylabel="RMSE", title="El error de Ridge según λ")
ax.legend()
plt.show()

El error de entrenamiento solo sube: penalizar empeora el ajuste a los datos que el modelo ve. El de prueba baja y después sube. El mejor $\lambda$, sin embargo, no se puede elegir mirando la prueba: eso sería hacer trampa otra vez. Se elige con validación cruzada, en la sección 11.

## 10. Lasso

**Objetivo:** ver cómo Lasso lleva coeficientes exactamente a 0 y elige variables.

Lasso cambia el castigo por el valor absoluto de los coeficientes:

$$\sum_i (y_i - \hat{y}_i)^2 + \lambda \sum_j |b_j|$$

Ese cambio hace que algunos coeficientes lleguen exactamente a 0. Una advertencia: scikit-learn escala el castigo de Lasso distinto que el de Ridge, así que los valores de `alpha` de uno y otro no se comparan. Con $\lambda = 30$:

In [ ]:
lasso = Lasso(alpha=30, max_iter=100000).fit(Z_ent, y_ent)
coef_lasso = pd.Series(lasso.coef_, index=X.columns)
print("variables con coeficiente distinto de 0:", (coef_lasso != 0).sum(), "de", len(coef_lasso))
coef_lasso[coef_lasso != 0].round(1)

In [ ]:
lambdas_lasso = np.logspace(-0.5, 2.6, 60)
camino_lasso = pd.DataFrame([Lasso(alpha=l, max_iter=100000).fit(Z_ent, y_ent).coef_ for l in lambdas_lasso],
                            index=lambdas_lasso, columns=X.columns)

fig, ax = plt.subplots(figsize=(9, 4.5))
for variable in X.columns:
    grueso = variable in ["turnos_carrera", "hits_carrera", "impulsadas_carrera"]
    ax.plot(camino_lasso.index, camino_lasso[variable], lw=3 if grueso else 1,
            label=variable if grueso else None)
ax.set_xscale("log")
ax.set(xlabel="λ (escala logarítmica)", ylabel="Coeficiente estandarizado", title="El camino de Lasso")
ax.legend()
plt.show()

## 11. Elegir λ con validación cruzada

**Objetivo:** elegir $\lambda$ sin tocar los datos de prueba.

La **validación cruzada** divide el entrenamiento en cinco partes. Para cada $\lambda$, ajusta el modelo con cuatro partes y mide el error en la quinta; lo repite cinco veces, cambiando la parte que valida, y promedia los cinco errores. Se elige el $\lambda$ con menor error promedio. `RidgeCV` y `LassoCV` hacen todo eso con `cv=5`:

In [ ]:
ridge_cv = RidgeCV(alphas=np.logspace(-2, 5, 200), cv=5).fit(Z_ent, y_ent)
print(f"λ elegido para Ridge: {ridge_cv.alpha_:.1f}")

In [ ]:
lasso_cv = LassoCV(cv=5, random_state=0, max_iter=100000).fit(Z_ent, y_ent)
coef_lasso_cv = pd.Series(lasso_cv.coef_, index=X.columns)
print(f"λ elegido para Lasso: {lasso_cv.alpha_:.1f}")
print("variables que conserva:", (coef_lasso_cv != 0).sum(), "de", len(coef_lasso_cv))
coef_lasso_cv[coef_lasso_cv != 0].round(1)

Recién ahora, con cada $\lambda$ ya elegido, se mide el error en los datos de prueba, una sola vez:

In [ ]:
resultados = pd.DataFrame({
    "RMSE en prueba": [rmse(y_pru, y_ent.mean()), rmse(y_pru, mco.predict(Z_pru)),
                       rmse(y_pru, ridge_cv.predict(Z_pru)), rmse(y_pru, lasso_cv.predict(Z_pru))],
    "variables usadas": [0, X.shape[1], X.shape[1], (coef_lasso_cv != 0).sum()],
}, index=["sin modelo (la media)", "mínimos cuadrados", "Ridge", "Lasso"])
resultados.round(0).astype(int)

Ridge y Lasso se equivocan menos que mínimos cuadrados en los jugadores que no vieron, y Lasso lo logra con 7 de las 19 variables. ¿Es suerte de este reparto? Se repite todo con 20 repartos distintos:

In [ ]:
filas = []
for semilla in range(1, 21):
    xe, xp, ye, yp = train_test_split(X, y, test_size=0.5, random_state=semilla)
    esc = StandardScaler().fit(xe)
    ze, zp = esc.transform(xe), esc.transform(xp)
    filas.append({
        "mínimos cuadrados": rmse(yp, LinearRegression().fit(ze, ye).predict(zp)),
        "Ridge": rmse(yp, RidgeCV(alphas=np.logspace(-2, 5, 200), cv=5).fit(ze, ye).predict(zp)),
        "Lasso": rmse(yp, LassoCV(cv=5, random_state=0, max_iter=100000).fit(ze, ye).predict(zp)),
    })
repartos = pd.DataFrame(filas)

print("RMSE promedio en prueba:")
print(repartos.mean().round(0).astype(int))
print("repartos en que Ridge gana a mínimos cuadrados:", (repartos["Ridge"] < repartos["mínimos cuadrados"]).sum(), "de 20")
print("repartos en que Lasso gana a mínimos cuadrados:", (repartos["Lasso"] < repartos["mínimos cuadrados"]).sum(), "de 20")

En 17 de los 20 repartos cada uno le gana a mínimos cuadrados: no es suerte. **¿Ridge o Lasso?** Ninguno es mejor siempre (James et al., p. 224). Lasso conviene si se sospecha que pocas variables importan, porque deja un modelo más simple; Ridge, si muchas aportan un poco o dicen casi lo mismo, porque estabiliza sus coeficientes sin descartar ninguna.

---

**Próxima clase (martes 22-sep), la última del módulo:** reducción de dimensionalidad (PCA) y las presentaciones orales del avance de cada equipo. El jueves 17 no hay clases, y el informe escrito de avance se entrega el viernes 25 de septiembre.